# Classification and regression as imputation

Use missing target columns to run supervised classification and regression through the same imputation API.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from mimic import MIMIC, RandomForestPathEncoder, MixedFeatureDecoder

Xc, yc = make_classification(n_samples=180, n_features=5, n_informative=4, n_redundant=0, random_state=1)
Xr, yr = make_regression(n_samples=180, n_features=1, noise=8, random_state=1)
df = pd.DataFrame(Xc, columns=[f"x{i}" for i in range(5)])
df["reg_target"] = yr
df["class_target"] = np.where(yc == 1, "positive", "negative")
train_idx, test_idx = train_test_split(df.index, test_size=0.25, random_state=1, stratify=df["class_target"])
work = df.copy()
work.loc[test_idx, ["reg_target", "class_target"]] = np.nan
work.head()

In [ ]:
mimic = MIMIC(
    regression_columns=["x0", "x1", "x2", "x3", "x4", "reg_target"],
    classification_columns=["class_target"],
    encoder=RandomForestPathEncoder(n_estimators=20, embedding_dim=8, random_state=1),
    decoder=MixedFeatureDecoder.random_forest(n_estimators=20, random_state=1),
    n_bootstrap=2,
    random_state=1,
)
mimic.fit(work)
pred = mimic.impute(work, columns=["reg_target", "class_target"])
y_reg_true = df.loc[test_idx, "reg_target"]
y_reg_pred = pred.loc[test_idx, "reg_target"].astype(float)
y_cls_true = df.loc[test_idx, "class_target"]
y_cls_pred = pred.loc[test_idx, "class_target"]
metrics = {
    "regression_rmse": np.sqrt(mean_squared_error(y_reg_true, y_reg_pred)),
    "regression_mae": mean_absolute_error(y_reg_true, y_reg_pred),
    "classification_accuracy": accuracy_score(y_cls_true, y_cls_pred),
    "classification_f1": f1_score(y_cls_true, y_cls_pred, pos_label="positive"),
}
metrics

In [ ]:
mimic.confidence(work.loc[test_idx], columns=["reg_target", "class_target"]).head()